# 03 – Task 2: Node Classification (community vs. non-community)

**Question:** Can a classifier correctly label nodes as community members
using only graph-structural features (without knowing the community directly)?

## Binary classification target
- `1` – node belongs to an overlapping community
- `0` – non-community node

## Feature sets tested
We compare many feature sets to understand what information is most useful:

| Feature set | Description |
|-------------|-------------|
| `lcc` | Local clustering coefficient (single feature) |
| `bc` | Betweenness centrality |
| `cc` | Closeness centrality |
| `dc` | Degree centrality |
| `ndc` | Normalised degree centrality |
| `ec` | Eigenvector centrality |
| `eccen` | Eccentricity |
| `core` | Core number (k-core) |
| `participation + in_mod_deg` | Community-aware baseline ("cheating") |
| `node2vec` | 16-dim node2vec embeddings |
| `struc2vec` | 16-dim struc2vec embeddings |

## Models
Logistic Regression, Random Forest, LightGBM, XGBoost

## Metrics
- **ROC-AUC** – area under ROC curve
- **Average Precision** – area under precision-recall curve
- **Average Precision at cutoff** – precision-recall at thresholds 0.01, 0.05, 0.1

In [ ]:
# ── CHANGE THIS to switch dataset ──────────────────────────────────────────
DATASET = "facebook"
# ───────────────────────────────────────────────────────────────────────────

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb
import xgboost as xgb

ROOT = Path("..")
PATH_COMBINED = ROOT / f"data/04_models/{DATASET}_combined.parquet"

# Thresholds for top-k precision evaluation
CUTOFFS = [0.01, 0.05, 0.1]

print(f"Dataset: {DATASET}")

## Step 1 – Load data and standardise features

Run `01_data_preparation.ipynb` first.

We standardise all features (zero mean, unit variance) before training —
this is important for Logistic Regression and helps embeddings-based models too.

In [ ]:
df = pd.read_parquet(PATH_COMBINED)

COMMUNITY_FREE  = ["lcc", "bc", "cc", "dc", "ndc", "ec", "eccen", "core"]
COMMUNITY_AWARE = ["participation", "in_mod_deg"]   # cheating baseline
N2V_COLS = [c for c in df.columns if c.startswith("emb_node2vec")]
S2V_COLS = [c for c in df.columns if c.startswith("emb_struc2vec")]
ALL_FEATURES = COMMUNITY_FREE + COMMUNITY_AWARE + N2V_COLS + S2V_COLS

# Standardise features
scaler = StandardScaler()
df[ALL_FEATURES] = scaler.fit_transform(df[ALL_FEATURES])

print(f"Nodes: {len(df)}")
print(f"Class distribution:\n{df['target'].value_counts().to_string()}")

## Step 2 – Stratified train / test split

Stratified = same class ratio in train and test.
70% train, 30% test.

In [ ]:
X = df[ALL_FEATURES]
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=2023, stratify=y
)
X_train = X_train.reset_index(drop=True)
X_test  = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test  = y_test.reset_index(drop=True)

print(f"Train: {len(X_train)} | Test: {len(X_test)}")
print(f"Train class ratio: {y_train.mean():.3f}")
print(f"Test  class ratio: {y_test.mean():.3f}")

## Step 3 – Define classifiers

In [ ]:
def make_classifiers():
    """Return dict of {name: unfitted classifier}."""
    return {
        "LogisticRegression": LogisticRegression(
            penalty="l2", C=1.0, class_weight="balanced",
            solver="lbfgs", max_iter=1000,
            fit_intercept=True, random_state=2023, n_jobs=-1
        ),

        "RandomForest": RandomForestClassifier(
            n_estimators=1000, criterion="gini",
            max_depth=10, min_samples_split=2,
            min_samples_leaf=1, n_jobs=-1, random_state=2023
        ),

        "LightGBM": lgb.LGBMClassifier(
            learning_rate=0.1, objective="binary",
            boosting_type="gbdt",
            n_estimators=1000,
            n_jobs=-1,
        ),

        "XGBoost": xgb.XGBClassifier(
            booster="gbtree", random_state=2023,
            objective="binary:logistic",
            base_score=0.5, verbosity=0,
            n_jobs=-1, eval_metric="auc",
        ),
    }

print("Classifiers:", list(make_classifiers().keys()))

## Step 4 – Define feature sets to evaluate

In [ ]:
# Each entry: (label_for_results, list_of_feature_columns)
FEATURE_SETS = (
    # Individual community-free features
    [(feat, [feat]) for feat in COMMUNITY_FREE]
    # Community-aware baseline ("cheating")
    + [("participation+in_mod_deg", COMMUNITY_AWARE)]
    # Embedding sets
    + [("node2vec", N2V_COLS)]
    + [("struc2vec", S2V_COLS)]
)

for label, cols in FEATURE_SETS:
    print(f"  {label:30s} → {len(cols)} column(s)")

## Step 5 – Train and evaluate all combinations

For each feature set × classifier we compute ROC-AUC and Average Precision.
We also compute Average Precision after thresholding predictions at 0.01 / 0.05 / 0.1.

In [ ]:
records = []

for feat_label, feat_cols in FEATURE_SETS:
    for clf_name, clf in make_classifiers().items():
        try:
            clf.fit(X_train[feat_cols], y_train)
            y_pred_proba = clf.predict_proba(X_test[feat_cols])[:, 1]

            roc_auc = roc_auc_score(y_test, y_pred_proba)
            avg_prec = average_precision_score(y_test, y_pred_proba)

            row = {
                "features"      : feat_label,
                "model"         : clf_name,
                "roc_auc"       : round(roc_auc, 4),
                "avg_precision" : round(avg_prec, 4),
            }
            # Average Precision at cutoff: binarise predictions at threshold
            for cutoff in CUTOFFS:
                y_binary = (y_pred_proba > cutoff).astype(int)
                row[f"avg_prec@{cutoff}"] = round(
                    average_precision_score(y_test, y_binary), 4
                )

            records.append(row)
            print(f"  {feat_label:30s} | {clf_name:20s} | "
                  f"AUC={roc_auc:.3f}  AP={avg_prec:.3f}")

        except Exception as e:
            records.append({
                "features": feat_label, "model": clf_name,
                "roc_auc": None, "avg_precision": None, "error": str(e)
            })
            print(f"  ERROR {feat_label} | {clf_name}: {e}")

results = pd.DataFrame(records)
print(f"\nDone. {len(results)} combinations evaluated.")

## Step 6 – Results

In [ ]:
# Full results sorted by ROC-AUC
results.sort_values("roc_auc", ascending=False)

In [ ]:
# Best model per feature set (by ROC-AUC)
best = (
    results.sort_values("roc_auc", ascending=False)
           .groupby("features")
           .first()[["model", "roc_auc", "avg_precision"]]
           .sort_values("roc_auc", ascending=False)
)
print("Best model per feature set:")
best

In [ ]:
# Pivot: feature sets as rows, models as columns (ROC-AUC)
pivot_auc = results.pivot_table(
    index="features", columns="model", values="roc_auc"
).round(4)
print("ROC-AUC by feature set and model:")
pivot_auc

In [ ]:
# Save results
PATH_OUT = ROOT / f"data/05_reporting/{DATASET}_task2.parquet"
results.to_parquet(PATH_OUT, index=False)
print(f"Saved to {PATH_OUT}")